# Thí nghiệm 4 — PhoGPT-4B-Chat sinh mã ICD-10 tự do

Bổ sung hàng thứ 10 cho Bảng 12: một LLM **bản ngữ tiếng Việt** chạy ở chế độ
không ràng buộc danh mục, sinh bất kỳ mã nào trong 13.081 mã của
Quyết định 4469/QĐ-BYT 2020.

Notebook này dựng theo đúng khuôn `thucnghiem3a_llama3_8b.ipynb` để kết quả
so trực tiếp được với ba LLM của TN3. **Prompt và bộ chấm là bản sao nguyên văn
của TN3**, không sửa một ký tự — đó là điều kiện để đưa vào cùng một bảng.

| Hit@1 PhoGPT | Ngụ ý | Ảnh hưởng bài |
|---|---|---|
| ≤ 5% | Kích cỡ model quan trọng hơn ngôn ngữ pre-training | Thêm bằng chứng cho Finding B/C |
| 5–15% | Rơi vào dải chung của 3 LLM kia | Thêm 1 câu ở §6.5, cover reviewer |
| ≥ 15% (vượt Llama-3 12,71%) | Vietnamese-native thắng | Reframe Finding C |

**Mốc đã có để đối chiếu** (đều tính trên cùng 181 ca, cùng bộ chấm):

| Hệ tự do | Hit@1 | Reachable | Out-of-reach | MRR | McNemar vs MedKG-HRR (n=118) |
|---|---|---|---|---|---|
| Llama-3-8B-Instruct | 12,71% | 17,80% | 3,17% | 0,1358 | b=20, c=5, p=0,0041 |
| Qwen2.5-7B-Instruct | 7,73% | 10,17% | 3,17% | 0,1076 | b=11, c=5, p=0,2101 |
| Phi-3.5-mini-instruct | 7,73% | 7,63% | **7,94%** | 0,0930 | b=9, c=6, p=0,6072 |
| BM25 tự do (13.081 mã) | 8,29% | 9,32% | 6,35% | 0,1128 | — |
| MedKG-HRR (of-record) | 3,31% | 5,08% | 0,00% | 0,0442 | — |

Mọi con số trong bảng trên đã được tính lại từ các tệp per-query trong
`so_lieu_cuoi_bo_sung_TN3` ngày 17/08/2026 và khớp với phiếu.

---

**Runtime:** `Runtime → Change runtime type → T4 GPU`.
PhoGPT-4B ở fp16 cần ~8 GB VRAM, T4 (16 GB) dư. Không cần Colab Pro, không cần token HF.

**Thời gian:** cài ~3 phút, tải model ~5–8 phút, chạy 181 ca ~20–40 phút trên T4.

## 1. Kiểm GPU

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), \
    "KHONG CO GPU. Vao Runtime -> Change runtime type -> T4 GPU roi chay lai."
p = torch.cuda.get_device_properties(0)
print(f"\nGPU : {p.name}")
print(f"VRAM: {p.total_memory/1e9:.2f} GB")

## 2. Cài thư viện — **ghim `transformers==4.44.2`, và phải Restart sau cell này**

PhoGPT dùng kiến trúc **MPT**, không phải Llama. Bắt buộc `trust_remote_code=True`
cho **cả ba** `AutoConfig`, `AutoTokenizer`, `AutoModelForCausalLM` — thiếu một cái là
lỗi ở đúng chỗ khó đọc nhất.

**Vì sao phải ghim bản cũ.** Mã kiến trúc MPT của PhoGPT viết cho `transformers` đời 2023
và không còn được cập nhật. Bản `transformers` mới làm nó gãy ở ít nhất hai chỗ:
`modeling_mpt.py` import ba lớp rotary mà đợt tái cấu trúc RoPE đã gỡ bỏ, và
`check_imports` đòi `triton_pre_mlir` cho một nhánh mã chết. Phần 6 có bản vá cho cả hai,
nhưng ghim `4.44.2` — bản cuối còn đủ ba lớp rotary — là đường chắc ăn hơn hẳn.

⚠ **Hai điều bắt buộc, sai là hỏng cả môi trường:**

1. Cell dưới **gỡ sạch** `transformers` cũ trước khi cài. Cài đè mà không gỡ sẽ để lại
   trạng thái **lẫn hai phiên bản** — triệu chứng là
   `ImportError: cannot import name 'is_flax_available' from 'transformers.utils'`,
   nghĩa là tệp của hai bản nằm chung một thư mục. Gặp lỗi đó thì chạy lại đúng cell này.
2. Chạy xong **phải `Runtime → Restart session`** rồi chạy tiếp từ Phần 3. Colab đã nạp
   sẵn `transformers` vào bộ nhớ; không restart thì Python vẫn dùng bản cũ trong RAM.
   **Đừng chạy lại cell này sau khi restart** — chạy lại cũng không sao, chỉ mất thêm 1 phút.

`einops` là phụ thuộc ẩn của mã MPT trên Hub; không cài thì `trust_remote_code`
báo `ModuleNotFoundError` giữa lúc tải trọng số.

In [ ]:
# Go sach truoc, roi moi cai. Cai de len ban khac se de lai trang thai lan hai phien ban.
%pip uninstall -y -q transformers tokenizers
%pip install -q "transformers==4.44.2" accelerate bitsandbytes einops python-docx

import transformers, torch
print("transformers:", transformers.__version__, "(can 4.44.2)")
print("torch       :", torch.__version__)
print("\n" + "=" * 68)
print("BAY GIO: Runtime -> Restart session, roi chay tiep tu Phan 3.")
print("=" * 68)

### 2b. Chốt phiên bản — chạy **sau khi đã Restart**

Cell này dừng ngay nếu môi trường chưa đúng, thay vì để bạn phát hiện ở phút thứ 20.

In [ ]:
import transformers, sys

V = transformers.__version__
print("transformers:", V)

if V != '4.44.2':
    raise SystemExit(
        f"transformers dang la {V}, can 4.44.2.\n"
        "  - Neu VUA chay cell Phan 2: bam Runtime -> Restart session roi chay lai cell nay.\n"
        "  - Neu chua chay: quay lai chay cell Phan 2 truoc.")

# Kiem gan: goi thu ham ma ban lan lon hay lam gay
try:
    from transformers.utils import is_flax_available  # chi co o 4.x
    from transformers.models.llama.modeling_llama import LlamaDynamicNTKScalingRotaryEmbedding
    print("Moi truong sach, du 3 lop rotary. Chay tiep Phan 3.")
except ImportError as e:
    raise SystemExit(
        f"Cai dat con lan hai phien ban ({e}).\n"
        "Chay lai cell Phan 2 (co buoc %pip uninstall) roi Restart session.")

## 3. Nạp dữ liệu đầu vào

Chạy cell dưới rồi chọn **`tn4_input.zip`** (hoặc `tn3_input.zip` — cùng nội dung).
Gói gồm 6 tệp:

| Tệp | Vai trò |
|---|---|
| `independent_scored_perquery.csv` | 181 ca độc lập + nhãn vàng + kết quả MedKG-HRR of-record |
| `ICD10_cleaned.csv` | 13.081 mã ICD-10 để validate |
| `tn1_perquery_*.csv` (4 tệp) | McNemar so với 4 baseline đóng danh mục |

Nếu chép thêm ba tệp `tn3_perquery_{llama3_8b,qwen25_7b,phi35_mini}.csv` vào gói thì
Phần 11 tự tính luôn McNemar PhoGPT vs từng LLM tự do của TN3. Không có cũng chạy được.

In [ ]:
import os, zipfile
from google.colab import files

os.makedirs('/content/tn4_data', exist_ok=True)
up = files.upload()
for ten in up:
    if ten.endswith('.zip'):
        with zipfile.ZipFile(ten) as z:
            z.extractall('/content/tn4_data')
    else:
        os.replace(ten, f'/content/tn4_data/{ten}')

print()
for f in sorted(os.listdir('/content/tn4_data')):
    print(' ', f)

## 4. Đọc và kiểm dữ liệu

Ba con số phải khớp: **181** ca, **13.081** mã, **118/63** reachable–out-of-reach.
Lệch bất kỳ con số nào thì cell tự dừng — chạy tiếp với dữ liệu sai còn tốn hơn.

In [ ]:
TAG      = 'phogpt_4b'
MODEL_ID = 'vinai/PhoGPT-4B-Chat'

import os
import pandas as pd

DATA = '/content/tn4_data'
OUT  = f'/content/tn4_ket_qua_{TAG}'
os.makedirs(OUT, exist_ok=True)

df_test = pd.read_csv(f'{DATA}/independent_scored_perquery.csv')
df_icd  = pd.read_csv(f'{DATA}/ICD10_cleaned.csv')

valid_codes_full  = set(df_icd['Mã ICD'].astype(str).str.strip().str.upper())
valid_codes_3char = set(c[:3] for c in valid_codes_full)

n_reach = int((df_test['voi_toi_duoc'] == True).sum())
n_outr  = int((df_test['voi_toi_duoc'] == False).sum())

print(f"So ca test          : {len(df_test):>6}   (ky vong 181)")
print(f"So ma ICD-10 day du : {len(valid_codes_full):>6}   (ky vong 13081)")
print(f"So phan nhom 3 ky tu: {len(valid_codes_3char):>6}")
print(f"Reachable / out     : {n_reach} / {n_outr}   (ky vong 118 / 63)")

assert len(df_test) == 181,            "SAI so ca test"
assert len(valid_codes_full) == 13081, "SAI so ma ICD"
assert (n_reach, n_outr) == (118, 63), "SAI ty le reachable"

oor    = df_test[df_test['voi_toi_duoc'] == False]
oor_in = sum(1 for g in oor['gold3'].astype(str) if g.strip().upper()[:3] in valid_codes_3char)
print(f"\nNhan out-of-reach nam trong catalogue ICD-10: {oor_in}/{len(oor)}")
print("=> nam ngoai tam voi cua DO THI, khong nam ngoai ICD-10." if oor_in == len(oor)
      else "=> CANH BAO: co nhan khong thuoc ICD-10, kiem lai buoc gan nhan.")

## 5. Prompt và bộ chấm — **bản sao nguyên văn của TN3**

Prompt dưới đây được chép nguyên văn từ cell 16 của `thucnghiem3a_llama3_8b.ipynb`,
không thêm không bớt. Phiếu TN4 có in lại một bản chép tay khác vài chữ (thiếu
`theo mẫu` ở quy tắc 4, và bọc thêm cặp thẻ `### Câu hỏi:` / `### Trả lời:`).
Bản notebook này giữ prompt TN3 gốc vì hai lý do:

1. Phiếu tự yêu cầu *"Dùng ĐÚNG prompt của TN3 để so được với 3 LLM khác. KHÔNG được modify"* —
   bản chép tay trong phiếu đã lệch so với TN3 thật.
2. Cặp thẻ `### Câu hỏi:` / `### Trả lời:` **không cần chèn tay**: PhoGPT-4B-Chat có
   `chat_template` trên Hub và `apply_chat_template` sẽ chèn đúng cặp thẻ đó.
   Chèn tay rồi lại chèn qua template là nhân đôi thẻ, model sinh kém hẳn.
   Phần 7 xử lý việc này có kiểm tra, có đường lui.

Metric chính: **Hit@1 ở cấp phân nhóm ICD-10 ba ký tự**, giống hệt TN1/TN2/TN3.

In [ ]:
PROMPT_TEMPLATE = """Bạn là bác sĩ trợ lý chuyên chẩn đoán bằng mã ICD-10.

NHIỆM VỤ: Đọc mô tả triệu chứng bệnh nhân, đưa ra danh sách TOP-10 mã ICD-10
có khả năng nhất, sắp xếp theo xác suất giảm dần.

QUY TẮC:
1. Mỗi mã ICD-10 phải là mã HỢP LỆ (định dạng: 1 chữ cái + 2-3 chữ số +
   tùy chọn "." + 1-2 chữ số. Ví dụ: A00, B07.9, K21.9).
2. Được phép dùng BẤT KỲ mã nào trong toàn bộ catalogue ICD-10 của
   Bộ Y tế Việt Nam (Quyết định 4469/QĐ-BYT 2020), gồm khoảng 13.081 mã.
3. Đưa mã 3 ký tự (phân nhóm) nếu không đủ tự tin về ký tự thứ 4.
4. KHÔNG giải thích. Chỉ trả về JSON theo mẫu.

VÍ DỤ:
Mô tả: "Em bị đau bụng nhiều, buồn nôn, sốt nhẹ, đã 3 ngày."
Kết quả JSON: {{"top10_icd": ["K35.9", "K52.9", "K37", "K59.0", "R10.4", "K85.9", "R11", "A09", "K80", "R50.9"]}}

---
Mô tả: "{query_text}"
Kết quả JSON:"""

# Bam prompt de chung minh trong goi tai lap rang TN4 dung dung prompt cua TN3
import hashlib
PROMPT_SHA = hashlib.sha256(PROMPT_TEMPLATE.encode('utf-8')).hexdigest()
print("SHA-256 prompt:", PROMPT_SHA)
print(PROMPT_TEMPLATE.format(query_text="<cau hoi nguoi benh>")[-320:])

In [ ]:
import json, re

def chuan_hoa_ma_icd(raw_code):
    """Bo khoang trang, dau cham cuoi, uppercase. None neu sai dinh dang."""
    if not raw_code:
        return None
    code = str(raw_code).strip().upper().rstrip('.')
    m = re.match(r'^([A-Z]\d{2}(?:\d)?(?:\.\d{1,2})?)$', code)
    return m.group(1) if m else None


def extract_json_safe(raw):
    """Trich JSON tu raw text, chiu duoc text thua truoc/sau."""
    m = re.search(r'\{[^{}]*"top10_icd"\s*:\s*\[[^\]]*\][^{}]*\}', raw)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return {'top10_icd': re.findall(r'\b([A-Z]\d{2}(?:\.\d{1,2})?)\b', raw)[:10]}


def cham_1_ca(llm_output_raw, gold3, gold_full, valid_3char):
    raw_list = extract_json_safe(llm_output_raw).get('top10_icd', [])
    top_10   = [c for c in (chuan_hoa_ma_icd(x) for x in raw_list) if c][:10]
    top_10_3 = [c[:3] for c in top_10]

    g3 = str(gold3).strip().upper()
    gf = str(gold_full).strip().upper()
    try:
        rank_3char = top_10_3.index(g3) + 1
    except ValueError:
        rank_3char = 0

    return {
        'top_10'              : ' || '.join(top_10),
        'top_10_3char'        : ' || '.join(top_10_3),
        'hit1'                : bool(top_10_3 and top_10_3[0] == g3),
        'hit5'                : g3 in top_10_3[:5],
        'hit10'               : g3 in top_10_3[:10],
        'hit1_full_icd'       : bool(top_10 and top_10[0] == gf),
        'rank_3char'          : rank_3char,
        'n_ma_parse_duoc'     : len(top_10),
        'n_valid_in_catalogue': sum(1 for c in top_10_3 if c in valid_3char),
    }


_r = cham_1_ca('{"top10_icd": ["K21.9","R10","XX","J32.","A09"]}',
               'R10', 'R10.4', valid_codes_3char)
assert _r['rank_3char'] == 2 and _r['n_ma_parse_duoc'] == 4, _r
print("Bo cham OK:", _r)

## 6. Nạp model — PhoGPT-4B-Chat

Cấu hình mặc định **fp16**, ~8 GB VRAM. Nếu T4 báo OOM (hiếm, nhưng có khi Colab đã
cấp một phần VRAM cho tiến trình khác) thì đổi `DUNG_4BIT = True` rồi chạy lại cell.

**Trước khi nạp phải vá phụ thuộc `triton_pre_mlir`** — không vá thì cell chết ngay ở
`AutoConfig.from_pretrained` với `ImportError: This modeling file requires the following
packages that were not found in your environment: triton_pre_mlir`.

Lý do, và vì sao bản vá dưới đây an toàn: PhoGPT là MPT, nên `trust_remote_code` kéo về
cả `flash_attn_triton.py`. `transformers` **kiểm tĩnh mọi import trong mọi tệp nó tải về**,
kể cả tệp không bao giờ chạy. Mà tệp đó đúng là không bao giờ chạy:

- `config.json` của PhoGPT đặt sẵn `attn_config.attn_impl = "torch"`, không phải `"triton"`;
- dòng `from .flash_attn_triton import flash_attn_func` nằm **bên trong hàm**
  `triton_flash_attn_fn` (attention.py dòng 174), chỉ gọi khi `attn_impl == 'triton'`.

Nên chỉ cần một module rỗng mang tên `triton_pre_mlir` để qua cửa kiểm tra. Không cần
build gói triton thật của MosaicML (nặng, hay hỏng trên Colab đời mới), và không có mã
triton nào được thực thi. Cell còn ghi đè `attn_impl = 'torch'` một lần nữa cho chắc.

**Cell vá còn dọn cache module động.** Một lần chạy hỏng ở bước trên để lại thư mục
`~/.cache/huggingface/modules/transformers_modules/…` **tồn tại nhưng thiếu tệp**.
Lần chạy sau `transformers` thấy thư mục đã có nên không chép lại, rồi chết ở
`FileNotFoundError: … flash_attn_triton.py`. Đây là lỗi khác hẳn lỗi đầu, dù cùng nhắc
một tên tệp. Xoá thư mục đó là nó chép lại đầy đủ; cell làm việc này mỗi lần chạy, vô hại
vì chỉ là mấy tệp `.py` nhỏ.

**Và vá ba lớp rotary.** `modeling_mpt.py` của PhoGPT import ở cấp module ba lớp
`LlamaRotaryEmbedding`, `LlamaLinearScalingRotaryEmbedding`,
`LlamaDynamicNTKScalingRotaryEmbedding` từ `transformers.models.llama.modeling_llama`.
Bản `transformers` đời mới đã **gỡ hai lớp scaling** sau đợt tái cấu trúc RoPE, nên nạp
model là `ImportError` ngay. Vá bằng lớp giả là an toàn vì ba lớp ấy **chỉ** được gọi
trong `gen_rotary_embedding` (modeling_mpt.py dòng 50–58), và hàm đó chỉ chạy khi
`attn_config['rope'] = True`. PhoGPT chạy **ALiBi** (`alibi = True`, `rope = False`), nên
không lớp nào được khởi tạo. Lớp giả cố tình **ném lỗi nếu bị khởi tạo thật**, để không
bao giờ hỏng âm thầm.

*Đường lui nếu vá không ăn:* ghim `%pip install -q "transformers==4.44.2"` rồi
`Runtime → Restart session`. Bản đó còn đủ ba lớp, chạy được mã gốc của PhoGPT không cần vá.

Ba chỗ khác TN3 và đều bắt buộc:

- **`trust_remote_code=True` cho cả ba lớp.** PhoGPT là MPT, mã kiến trúc nằm trên Hub.
- **`config.init_device = "cuda"`** — theo đúng hướng dẫn chính thức của VinAI. Không đặt
  thì trọng số khởi tạo trên CPU rồi mới chuyển, chậm và tốn RAM hệ thống.
- **`TERMINATORS`** — vẫn cần, dò cả `<|endoftext|>` lẫn `</s>`. Không có thì model sinh
  đủ `max_new_tokens` ở mọi ca: chậm gấp nhiều lần và viết thêm rác sau JSON.

In [ ]:
# ===== VA TRUOC KHI NAP MODEL - phai chay TRUOC AutoConfig =====
import os, sys, shutil, tempfile

# --- (1) Don cache module dong dang do cua transformers ---
# Lan chay hong truoc do de lai thu muc transformers_modules TON TAI nhung THIEU tep.
# transformers thay thu muc da co nen khong chep lai, roi chet o
# FileNotFoundError ... flash_attn_triton.py. Xoa di la no chep lai day du.
try:
    from transformers.utils import HF_MODULES_CACHE as _MOD_CACHE
except Exception:
    _MOD_CACHE = os.path.expanduser('~/.cache/huggingface/modules')

_tm = os.path.join(_MOD_CACHE, 'transformers_modules')
if os.path.isdir(_tm):
    shutil.rmtree(_tm, ignore_errors=True)
    print("Da xoa cache module dong:", _tm)
else:
    print("Cache module dong sach san:", _tm)

# Bo cac module da nap do dang trong phien nay, neu khong Python van dung ban cu
for _m in [m for m in list(sys.modules) if m.startswith('transformers_modules')]:
    del sys.modules[_m]

# --- (2) Stub triton_pre_mlir ---

try:
    import triton_pre_mlir  # noqa: F401
    print("triton_pre_mlir: da co san")
except ImportError:
    goc = '/content' if os.path.isdir('/content') else tempfile.gettempdir()
    stub = os.path.join(goc, '_stub_triton')
    os.makedirs(os.path.join(stub, 'triton_pre_mlir'), exist_ok=True)
    with open(os.path.join(stub, 'triton_pre_mlir', '__init__.py'), 'w',
              encoding='utf-8') as f:
        f.write('# Stub rong. PhoGPT chay attn_impl="torch" nen nhanh triton\n'
                '# (flash_attn_triton.py) khong bao gio duoc import luc chay.\n'
                '# Module nay chi ton tai de qua buoc check_imports cua transformers.\n')
    if stub not in sys.path:
        sys.path.insert(0, stub)
    import triton_pre_mlir  # noqa: F401
    print("triton_pre_mlir: da tao stub tai", stub)

# --- (3) Va 3 lop rotary ma transformers doi moi da go bo ---
# modeling_mpt.py cua PhoGPT import LlamaRotaryEmbedding / LlamaLinearScaling... /
# LlamaDynamicNTKScaling... o CAP MODULE. Ban transformers moi khong con hai lop
# scaling => ImportError ngay khi nap model.
# An toan vi ba lop nay CHI duoc dung trong gen_rotary_embedding (modeling_mpt.py
# dong 50-58), ma ham do chi chay khi attn_config['rope'] = True. PhoGPT dung ALiBi
# (alibi=True, rope=False) nen khong lop nao duoc khoi tao. Lop gia duoi day nem loi
# neu bi khoi tao that - de khong bao gio hong am tham.
try:
    import transformers.models.llama.modeling_llama as _ll
except ImportError as _e:
    raise SystemExit(
        "Khong import duoc transformers.models.llama.modeling_llama: " + str(_e) + "\n"
        "Day KHONG phai loi cua PhoGPT ma la cai dat transformers dang lan hai phien ban.\n"
        "Xu ly: chay lai cell Phan 2 (co buoc %pip uninstall) roi Restart session.")

def _tao_lop_gia(ten):
    def __init__(self, *a, **k):
        raise RuntimeError(
            ten + " da bi go khoi ban transformers nay va lop gia vua bi khoi tao. "
            "PhoGPT le ra chay ALiBi (rope=False) nen khong duoc goi toi day. "
            "Xu ly: cai transformers==4.44.2 roi Restart session.")
    return type(ten, (object,), {'__init__': __init__})

_da_va = []
for _ten in ('LlamaRotaryEmbedding',
             'LlamaLinearScalingRotaryEmbedding',
             'LlamaDynamicNTKScalingRotaryEmbedding'):
    if not hasattr(_ll, _ten):
        setattr(_ll, _ten, _tao_lop_gia(_ten))
        _da_va.append(_ten)
print("Lop rotary da va:", ', '.join(_da_va) if _da_va else "khong thieu lop nao")

In [ ]:
DUNG_4BIT = False          # doi thanh True neu OOM

from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print(f"Dang tai: {MODEL_ID}  ({'4-bit NF4' if DUNG_4BIT else 'fp16'})")

config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
config.init_device = 'cuda'

# Chac chan khong roi vao nhanh triton (config goc da la 'torch', ghi de cho chac)
try:
    if isinstance(config.attn_config, dict):
        config.attn_config['attn_impl'] = 'torch'
    else:
        config.attn_config.attn_impl = 'torch'
    print("attn_impl =", config.attn_config['attn_impl']
          if isinstance(config.attn_config, dict) else config.attn_config.attn_impl)
except AttributeError:
    print("CANH BAO: khong doc duoc attn_config, bo qua buoc ghi de.")

kwargs = dict(config=config, trust_remote_code=True)
if DUNG_4BIT:
    kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16)
else:
    kwargs['torch_dtype'] = torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

TERMINATORS = [tokenizer.eos_token_id]
for t in ('<|endoftext|>', '</s>', '<|im_end|>', '<|eot_id|>'):
    i = tokenizer.convert_tokens_to_ids(t)
    if isinstance(i, int) and i >= 0 and i != tokenizer.unk_token_id and i not in TERMINATORS:
        TERMINATORS.append(i)

CO_CHAT_TEMPLATE = bool(getattr(tokenizer, 'chat_template', None))
print("Nap model xong!")
print("chat_template :", "CO -> dung apply_chat_template" if CO_CHAT_TEMPLATE
                          else "KHONG -> boc tay bang the ### Cau hoi / ### Tra loi")
print("TERMINATORS   :", TERMINATORS)

## 7. Hàm sinh

PhoGPT-4B-Chat được huấn luyện với khuôn `### Câu hỏi:\n...\n### Trả lời:`.
Hàm dưới ưu tiên `apply_chat_template` (đúng như TN3 làm với Llama-3) và **chỉ** bọc
thẻ bằng tay khi bản tokenizer tải về không kèm template. Hai đường đều cho ra cùng
một chuỗi thẻ, nên không lệch so với TN3 về mặt phương pháp.

Giải mã tham lam (`do_sample=False`) nên chạy lại cho đúng cùng kết quả.

In [ ]:
KHUON_PHOGPT = "### Câu hỏi:\n{noi_dung}\n### Trả lời:"

def sinh_predictions(query_text, max_new_tokens=250):
    prompt = PROMPT_TEMPLATE.format(query_text=query_text)

    if CO_CHAT_TEMPLATE:
        text = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True,
        )
    else:
        text = KHUON_PHOGPT.format(noi_dung=prompt)

    inputs = tokenizer(text, return_tensors="pt",
                       add_special_tokens=False).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=TERMINATORS,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip()


_thu = sinh_predictions(df_test.iloc[0]['query_text'])
print("Output thu nghiem:")
print(_thu[:400])
assert _thu, "Model tra ve rong. Kiem lai TERMINATORS va chat_template o Phan 6."

## 8. Dry run 10 ca — **bắt buộc, không được bỏ**

Bốn cổng của mục 6.1 phiếu. Fail bất kỳ cổng nào thì **dừng và báo thầy**,
không chạy full: 181 ca mất 20–40 phút, phát hiện lỗi prompt sau đó là mất trắng.

| Cổng | Ngưỡng | Nếu fail |
|---|---|---|
| JSON parse thành công | ≥ 8/10 | Prompt bug — kiểm `chat_template` ở Phần 6 |
| Mã hợp lệ trong catalogue | ≥ 70% | Model chế mã giả — thử tăng `max_new_tokens` |
| Có ≥ 1 mã hợp lý y khoa | đọc mắt | Model không hiểu prompt |
| Không CUDA OOM | không warning | Bật `DUNG_4BIT = True` |

In [ ]:
import time

print("=" * 70)
print(f"DRY RUN 10 CA - {MODEL_ID}")
print("=" * 70)

# Danh sach ma trong VI DU cua prompt - de phat hien model chep lai thay vi suy luan
MA_VI_DU_3 = ['K35', 'K52', 'K37', 'K59', 'R10', 'K85', 'R11', 'A09', 'K80', 'R50']

def ty_le_trung(top_10_3char, bo_ma):
    """Ty le ma trong output nam trong bo ma cua vi du.

    So theo TAP HOP chu khong theo vi tri: model co the sinh hong ky tu dau
    (vi du '35.9' thay vi 'K35.9') lam ca danh sach lech mot o, nhung ban chat
    van la chep. So theo vi tri se bo sot dung truong hop do.
    """
    ds = [c.strip() for c in str(top_10_3char).split('||') if c.strip()]
    if len(ds) < 3:          # duoi 3 ma thi khong du co so ket luan
        return 0.0
    return sum(1 for c in ds if c in set(bo_ma)) / len(ds)


def chep_vi_du(top_10_3char):
    """True neu >= 80% ma sinh ra deu nam trong vi du -> model dang chep."""
    return ty_le_trung(top_10_3char, MA_VI_DU_3) >= 0.8

t0, tong_valid, tong_ma, n_parse_ok, n_hit1, n_chep = time.time(), 0, 0, 0, 0, 0
for _, row in df_test.head(10).iterrows():
    raw = sinh_predictions(row['query_text'])
    sc  = cham_1_ca(raw, row['gold3'], row['gold'], valid_codes_3char)
    tong_valid += sc['n_valid_in_catalogue']
    tong_ma    += max(sc['n_ma_parse_duoc'], 1)
    n_parse_ok += 1 if sc['n_ma_parse_duoc'] > 0 else 0
    n_hit1     += 1 if sc['hit1'] else 0
    la_chep     = chep_vi_du(sc['top_10_3char'])
    n_chep     += 1 if la_chep else 0

    print(f"\n[{row['case_id']}] gold3={row['gold3']} | out-of-reach={not row['voi_toi_duoc']}")
    print(f"  hoi   : {str(row['query_text'])[:90]}...")
    print(f"  raw   : {raw[:150]}")
    print(f"  top10 : {sc['top_10_3char']}")
    print(f"  Hit@1={sc['hit1']}  rank={sc['rank_3char']}  "
          f"valid={sc['n_valid_in_catalogue']}/{sc['n_ma_parse_duoc']}"
          f"{'   *** CHEP LAI VI DU TRONG PROMPT ***' if la_chep else ''}")

ty_le    = 100 * tong_valid / tong_ma
giay     = (time.time() - t0) / 10
ty_le_chep = 100 * n_chep / 10

print("\n" + "=" * 70)
print(f"JSON parse ra >=1 ma : {n_parse_ok}/10   (cong: >= 8)")
print(f"Ty le ma hop le      : {ty_le:.1f}%      (cong: >= 70)")
print(f"CHEP VI DU trong prompt: {n_chep}/10   (cong: <= 2)")
print(f"Hit@1 tren 10 ca     : {n_hit1}/10")
print(f"Toc do               : {giay:.1f} giay/ca -> uoc {giay*181/60:.0f} phut cho 181 ca")

cong_qua = (n_parse_ok >= 8) and (ty_le >= 70) and (n_chep <= 2)
if cong_qua:
    print("\nOK, chay tiep Phan 9.")
else:
    print("\nFAIL CONG. DUNG LAI, khong chay full (muc 10.1 phieu).")
    if n_chep > 2:
        print("  -> Ly do: model CHEP LAI danh sach ma trong VI DU cua prompt thay vi")
        print("     doc mo ta benh nhan. Hit@1 do trong trang thai nay VO NGHIA:")
        print("     ca nao co gold trung mot ma cua vi du se 'dung' hoan toan ngau nhien.")
        print("     Chay Phan 8b de chan doan nguyen nhan truoc khi bao thay.")

## 8b. Chẩn đoán khi model chép ví dụ — **chỉ chạy khi cổng "chép ví dụ" FAIL**

Nếu Phần 8 báo model chép lại danh sách mã trong ví dụ, đừng vội kết luận
"PhoGPT không làm được task". Có ba khả năng rất khác nhau, và ba phép thử dưới đây
tách được chúng. Tốn khoảng 3 phút.

| Khả năng | Phép thử phát hiện | Ý nghĩa |
|---|---|---|
| Prompt bị bọc sai (thẻ đôi, cắt cụt) | Thử 1 — in nguyên văn chuỗi sau `apply_chat_template` | Lỗi kỹ thuật của ta, sửa được |
| Model bám ví dụ, bỏ qua mô tả | Thử 2 — **đổi ví dụ sang bộ mã khác**; nếu output đổi theo thì đã chắc | Tính chất của model |
| Model không hiểu task, ví dụ chỉ là chỗ bám | Thử 3 — bỏ hẳn ví dụ (zero-shot) | Tính chất của model |

Thử 2 là phép quyết định: nếu đổi ví dụ mà output chép theo ví dụ mới, thì chắc chắn model
đang sao chép chứ không suy luận, không còn nghi ngờ gì về lỗi kỹ thuật phía ta.

In [ ]:
print("=" * 72)
print("THU 1 - Chuoi thuc su dua vao model (kiem the co bi doi khong)")
print("=" * 72)
_p = PROMPT_TEMPLATE.format(query_text=df_test.iloc[0]['query_text'])
if CO_CHAT_TEMPLATE:
    _txt = tokenizer.apply_chat_template([{"role": "user", "content": _p}],
                                         tokenize=False, add_generation_prompt=True)
else:
    _txt = KHUON_PHOGPT.format(noi_dung=_p)

print("--- 200 ky tu DAU ---")
print(repr(_txt[:200]))
print("\n--- 200 ky tu CUOI ---")
print(repr(_txt[-200:]))
print(f"\nSo token dau vao: {len(tokenizer(_txt).input_ids)}")
for _the in ['### Câu hỏi:', '### Trả lời:']:
    print(f"  {_the!r} xuat hien {_txt.count(_the)} lan"
          f"{'  <-- LOI: the doi!' if _txt.count(_the) > 1 else ''}")

In [ ]:
print("=" * 72)
print("THU 2 - DOI VI DU sang bo ma khac han (phep quyet dinh)")
print("=" * 72)

# Vi du thay the: cung dinh dang, ma hoan toan khac, chu de khac (da lieu / mat)
PROMPT_DOI_VD = PROMPT_TEMPLATE.replace(
    'Mô tả: "Em bị đau bụng nhiều, buồn nôn, sốt nhẹ, đã 3 ngày."',
    'Mô tả: "Da tay em nổi mẩn đỏ, ngứa nhiều, bong tróc đã 2 tuần."'
).replace(
    '{{"top10_icd": ["K35.9", "K52.9", "K37", "K59.0", "R10.4", "K85.9", "R11", "A09", "K80", "R50.9"]}}',
    '{{"top10_icd": ["L20.9", "L23.9", "L30.9", "L21.9", "B35.1", "L28.0", "L40.9", "L50.9", "L85.3", "L98.9"]}}'
) if '{{' in PROMPT_TEMPLATE else PROMPT_TEMPLATE
assert 'L20.9' in PROMPT_DOI_VD, "Khong thay the duoc vi du - kiem lai chuoi PROMPT_TEMPLATE"

MA_VD_MOI_3 = ['L20', 'L23', 'L30', 'L21', 'B35', 'L28', 'L40', 'L50', 'L85', 'L98']

def sinh_voi_prompt(tpl, query_text, max_new_tokens=250):
    p = tpl.format(query_text=query_text)
    txt = (tokenizer.apply_chat_template([{"role": "user", "content": p}],
                                         tokenize=False, add_generation_prompt=True)
           if CO_CHAT_TEMPLATE else KHUON_PHOGPT.format(noi_dung=p))
    inp = tokenizer(txt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                             eos_token_id=TERMINATORS, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:],
                            skip_special_tokens=True).strip()

n_theo_vd_moi = 0
for _, row in df_test.head(5).iterrows():
    raw = sinh_voi_prompt(PROMPT_DOI_VD, row['query_text'])
    sc  = cham_1_ca(raw, row['gold3'], row['gold'], valid_codes_3char)
    theo = ty_le_trung(sc['top_10_3char'], MA_VD_MOI_3) >= 0.8
    n_theo_vd_moi += 1 if theo else 0
    print(f"  {row['case_id']}: {sc['top_10_3char'][:70]}"
          f"{'   <-- CHEP VI DU MOI' if theo else ''}")

print(f"\nSo ca chep theo vi du MOI: {n_theo_vd_moi}/5")
print("=> XAC NHAN: model sao chep vi du, khong doc mo ta benh nhan."
      if n_theo_vd_moi >= 4 else
      "=> Chua du 4/5. DOC BANG MAT 5 dong tren truoc khi ket luan: output ngan\n"
      "   (3-4 ma) van co the la duoi cua vi du moi. Neu moi ma sinh ra deu thuoc\n"
      "   bo L20/L23/L30/L21/B35/L28/L40/L50/L85/L98 thi VAN LA CHEP.")

In [ ]:
print("=" * 72)
print("THU 3 - BO HAN VI DU (zero-shot)")
print("=" * 72)

PROMPT_ZERO = """Bạn là bác sĩ trợ lý chuyên chẩn đoán bằng mã ICD-10.

NHIỆM VỤ: Đọc mô tả triệu chứng bệnh nhân, đưa ra danh sách TOP-10 mã ICD-10
có khả năng nhất, sắp xếp theo xác suất giảm dần.

QUY TẮC:
1. Mỗi mã ICD-10 phải là mã HỢP LỆ (định dạng: 1 chữ cái + 2-3 chữ số +
   tùy chọn "." + 1-2 chữ số. Ví dụ: A00, B07.9, K21.9).
2. Được phép dùng BẤT KỲ mã nào trong toàn bộ catalogue ICD-10 của
   Bộ Y tế Việt Nam (Quyết định 4469/QĐ-BYT 2020), gồm khoảng 13.081 mã.
3. Đưa mã 3 ký tự (phân nhóm) nếu không đủ tự tin về ký tự thứ 4.
4. KHÔNG giải thích. Chỉ trả về JSON dạng {{"top10_icd": [...]}}.

---
Mô tả: "{query_text}"
Kết quả JSON:"""

n_hit_zero, n_parse_zero = 0, 0
for _, row in df_test.head(5).iterrows():
    raw = sinh_voi_prompt(PROMPT_ZERO, row['query_text'])
    sc  = cham_1_ca(raw, row['gold3'], row['gold'], valid_codes_3char)
    n_hit_zero   += 1 if sc['hit1'] else 0
    n_parse_zero += 1 if sc['n_ma_parse_duoc'] > 0 else 0
    print(f"  {row['case_id']} (gold3={row['gold3']}): {sc['top_10_3char'][:70]}")
    print(f"     raw: {raw[:110]}")

print(f"\nZero-shot: parse duoc {n_parse_zero}/5, Hit@1 {n_hit_zero}/5")
print("\n" + "=" * 72)
print("KET LUAN GUI THAY - ghi lai 4 con so:")
print(f"  1. Chep vi du goc (Phan 8)        : {n_chep}/10")
print(f"  2. Chep vi du MOI (Thu 2)          : {n_theo_vd_moi}/5")
print(f"  3. Zero-shot parse duoc (Thu 3)    : {n_parse_zero}/5")
print(f"  4. Zero-shot Hit@1 (Thu 3)         : {n_hit_zero}/5")
print("=" * 72)

### Đọc kết quả chẩn đoán

**Nếu Thử 2 xác nhận chép ví dụ (≥ 4/5 ca chép theo ví dụ mới):** đây là tính chất của
PhoGPT-4B-Chat với prompt one-shot, không phải lỗi cấu hình của ta. Lúc này **phải báo thầy
và chờ quyết định**, vì mọi đường đi tiếp đều đụng vào tính so sánh được của Bảng 12:

| Đường | Được gì | Mất gì |
|---|---|---|
| Báo kịch bản A với ghi chú "model sao chép ví dụ" | Trung thực, không sửa prompt | Con số Hit@1 không đo được năng lực thật |
| Chuyển sang prompt zero-shot cho **riêng** PhoGPT | Có thể đo được năng lực thật | Khác prompt với 3 LLM kia → **phá điều kiện so sánh của phiếu** |
| Chuyển sang prompt zero-shot cho **cả 4 model** | Giữ so sánh công bằng | Phải chạy lại toàn bộ TN3, tốn vài giờ |

Phiếu TN4 mục 4 nói rõ *"KHÔNG được modify"* prompt, nên **không tự ý đổi**. Gửi thầy bốn
con số ở cell trên kèm 3 ví dụ output thô và để thầy chọn đường.

**Nếu Thử 2 không xác nhận** thì nguyên nhân nằm chỗ khác — gửi toàn bộ output của cả ba
phép thử, đừng chạy full.

## 9. Chạy full 181 ca

In [ ]:
from tqdm.auto import tqdm

t0, results = time.time(), []
for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=f"181 ca - {TAG}"):
    raw = sinh_predictions(row['query_text'])
    sc  = cham_1_ca(raw, row['gold3'], row['gold'], valid_codes_3char)
    results.append({
        'case_id'       : row['case_id'],
        'source_channel': row['source_channel'],
        'gold3'         : str(row['gold3']).strip().upper(),
        'gold_full'     : str(row['gold']).strip().upper(),
        'voi_toi_duoc'  : row['voi_toi_duoc'],
        'llm_raw'       : raw,
        **sc,
    })

df_out = pd.DataFrame(results)
PHUT_CHAY = (time.time() - t0) / 60
F_PQ = f'{OUT}/tn4_perquery_phogpt_unconstrained.csv'
df_out.to_csv(F_PQ, index=False, encoding='utf-8-sig')

print(f"\nXong {len(df_out)} ca trong {PHUT_CHAY:.1f} phut")
print(f"VRAM dinh             : {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print(f"So ca parse duoc 0 ma : {(df_out['n_ma_parse_duoc'] == 0).sum()}")
print(f"Da luu                : {F_PQ}")

## 10. Chỉ số tổng hợp

Dòng `hit1_outreach_pct` là con số quyết định: trên 63 ca này năm hệ đóng danh mục của TN1
đều đạt 0,00% **theo thiết kế**. Mốc cao nhất hiện có là Phi-3.5-mini với 7,94%.

In [ ]:
TEN_HE = 'PhoGPT-4B-Chat_unconstrained'
from scipy import stats

def wilson_ci(k, n, alpha=0.05):
    if n == 0:
        return (0.0, 0.0)
    z = stats.norm.ppf(1 - alpha / 2)
    p = k / n
    den = 1 + z**2 / n
    c = (p + z**2 / (2*n)) / den
    h = z * ((p*(1-p)/n + z**2/(4*n**2))**0.5) / den
    return (max(0, (c-h)*100), min(100, (c+h)*100))


n     = len(df_out)
reach = df_out[df_out['voi_toi_duoc'] == True]
outr  = df_out[df_out['voi_toi_duoc'] == False]

hit1_n     = int(df_out['hit1'].sum())
lo,   hi   = wilson_ci(hit1_n, n)
lo_o, hi_o = wilson_ci(int(outr['hit1'].sum()), len(outr))
mrr = df_out['rank_3char'].apply(lambda x: 1/x if x > 0 else 0).mean()

summary = {
    'system'              : TEN_HE,
    'model_id'            : MODEL_ID,
    'n'                   : n,
    'hit1_pct'            : round(100*hit1_n/n, 2),
    'hit1_ci'             : f'[{lo:.2f}, {hi:.2f}]',
    'hit5_pct'            : round(100*df_out['hit5'].sum()/n, 2),
    'hit10_pct'           : round(100*df_out['hit10'].sum()/n, 2),
    'mrr'                 : round(float(mrr), 4),
    'n_reachable'         : len(reach),
    'hit1_reachable_pct'  : round(100*reach['hit1'].sum()/len(reach), 2),
    'n_outreach'          : len(outr),
    'hit1_outreach_pct'   : round(100*outr['hit1'].sum()/len(outr), 2),
    'hit1_outreach_ci'    : f'[{lo_o:.2f}, {hi_o:.2f}]',
    'hit1_full_icd_pct'   : round(100*df_out['hit1_full_icd'].sum()/n, 2),
    'avg_n_valid_in_top10': round(float(df_out['n_valid_in_catalogue'].mean()), 2),
    'pct_ma_hop_le'       : round(100*df_out['n_valid_in_catalogue'].sum()
                                  / max(df_out['n_ma_parse_duoc'].sum(), 1), 2),
    'so_luong_bit'        : '4bit_nf4' if DUNG_4BIT else 'fp16',
    'vram_dinh_GB'        : round(torch.cuda.max_memory_allocated()/1e9, 2),
    'phut_chay'           : round(PHUT_CHAY, 1),
    'prompt_sha256'       : PROMPT_SHA,
}
pd.DataFrame([summary]).to_csv(f'{OUT}/tn4_summary.csv', index=False, encoding='utf-8-sig')
for k, v in summary.items():
    print(f"{k:<22}: {v}")

h = summary['hit1_pct']
KICH_BAN = ('A (<=5%)'  if h <= 5  else
            'B (5-15%)' if h < 15 else
            'C (>=15%)')
print("\n" + "=" * 68)
print("KICH BAN A (<=5%)   : kich co model quan trong hon ngon ngu pre-training" if h <= 5 else
      "KICH BAN B (5-15%)  : PhoGPT roi vao dai chung cua 3 LLM kia"            if h < 15 else
      "KICH BAN C (>=15%)  : Vietnamese-native thang -> PHAI reframe Finding C")
print("Moc: Llama-3 12.71% | Qwen2.5 7.73% | Phi-3.5 7.73% | BM25 tu do 8.29%")
print("=" * 68)

## 11. McNemar ghép cặp

Hai con số phiếu yêu cầu là **n=181** và **n=118 (chỉ reachable)**. Mốc đối chiếu:
Llama-3 trên n=118 cho b=20, c=5, p=0,0041.

Lưu ý kỹ thuật: bảng McNemar phải ghép cặp **theo `case_id`**, không theo thứ tự dòng.
Hai tệp có cùng 181 dòng nhưng khác thứ tự là chuyện thường; ghép theo vị trí sẽ cho ra
b/c sai mà không báo lỗi gì. Hàm dưới `reindex` theo `case_id` trước khi so.

In [ ]:
from scipy.stats import binomtest

med  = pd.read_csv(f'{DATA}/independent_scored_perquery.csv').set_index('case_id')
mine = df_out.set_index('case_id')
u_hit = mine['hit1'].astype(bool).astype(int).reindex(med.index).fillna(0).astype(int)
mask_reach = (med['voi_toi_duoc'] == True).reindex(med.index).fillna(False)

def mcnemar(other_hit, ten, mask=None):
    o = other_hit.reindex(med.index).fillna(0).astype(int)
    u = u_hit
    if mask is not None:
        o, u = o[mask], u[mask]
    b = int(((u == 1) & (o == 0)).sum())
    c = int(((u == 0) & (o == 1)).sum())
    p = binomtest(min(b, c), n=b+c, p=0.5).pvalue if (b+c) > 0 else 1.0
    return {'doi_chieu': ten, 'n': int(len(u)),
            'b_chi_phogpt_dung': b, 'c_chi_he_kia_dung': c,
            'ca_hai_dung': int(((u == 1) & (o == 1)).sum()),
            'mcnemar_p': round(float(p), 4)}

medkg_hit = (med['rank3'] == 1).astype(int)
rows = [
    mcnemar(medkg_hit, 'MedKG-HRR (of-record) — n=181'),
    mcnemar(medkg_hit, 'MedKG-HRR (of-record) — n=118 reachable', mask=mask_reach),
]

for fn, ten in [('tn1_perquery_bm25_only.csv',        'BM25-only (dong danh muc)'),
                ('tn1_perquery_retrieval_only.csv',   'Retrieval-only'),
                ('tn1_perquery_kg_only.csv',          'KG-only'),
                ('tn1_perquery_phi_35_zero_shot.csv', 'Phi-3.5 zero-shot (dong danh muc)'),
                ('tn3_perquery_llama3_8b.csv',        'Llama-3-8B tu do (TN3)'),
                ('tn3_perquery_qwen25_7b.csv',        'Qwen2.5-7B tu do (TN3)'),
                ('tn3_perquery_phi35_mini.csv',       'Phi-3.5-mini tu do (TN3)')]:
    fp = f'{DATA}/{fn}'
    if not os.path.exists(fp):
        print('THIEU (bo qua):', fn); continue
    oh = pd.read_csv(fp).set_index('case_id')['hit1'].astype(bool).astype(int)
    rows.append(mcnemar(oh, ten))

df_mc = pd.DataFrame(rows)
df_mc.to_csv(f'{OUT}/tn4_mcnemar.csv', index=False, encoding='utf-8-sig')
print(df_mc.to_string(index=False))

## 12. Phân tách theo chương ICD-10 và 63 ca out-of-reach

Không nằm trong danh mục nộp bắt buộc, nhưng TN3 có bảng này nên TN4 phải có để
xếp cạnh nhau được. Mốc BM25 tự do: chương R 11,1% (3/27), chương Z 0,0% (0/27).

In [ ]:
df_out['chuong'] = df_out['gold3'].astype(str).str[0]

g = df_out.groupby('chuong').agg(n=('hit1', 'size'), dung=('hit1', 'sum'))
g['hit1_pct'] = (100 * g['dung'] / g['n']).round(1)
print("Hit@1 theo chuong ICD-10 (chi chuong co >= 5 ca):")
print(g[g['n'] >= 5].sort_values('hit1_pct', ascending=False).to_string())
g.to_csv(f'{OUT}/tn4_theo_chuong.csv', encoding='utf-8-sig')

w = df_out[(df_out['voi_toi_duoc'] == False) & (df_out['hit1'] == True)]
print(f"\n{len(w)} ca OUT-OF-REACH ma PhoGPT bat dung "
      f"(5 he dong danh muc deu 0.00%; moc cao nhat la Phi-3.5 voi 5/63 = 7.94%):")
for _, r in w.iterrows():
    print(f"  {r['case_id']} | gold3={r['gold3']} | top1={str(r['top_10']).split(' || ')[0]}")
    print(f"     {str(med.loc[r['case_id'], 'query_text'])[:110]}")

## 13. Bảng tổng hợp để dán vào §6.5 / Bảng 12

Bảng 12 sau khi thêm hàng này có 10 hệ. Sáu hàng đầu là số đã chốt của TN1/TN3,
chép nguyên từ `tn3_bang_tong_hop_llama3_8b.csv`.

In [ ]:
bang = pd.DataFrame([
    {'Hệ thống': 'MedKG-HRR (đề xuất)',    'Hit@1 (%)': 3.31, 'KTC 95%': '[1,53; 7,04]',
     'Hit@5 (%)': 6.63,  'MRR': 0.0442, 'Reachable (%)': 5.08, 'Out-of-reach (%)': 0.00},
    {'Hệ thống': 'Retrieval-only',         'Hit@1 (%)': 6.08, 'KTC 95%': '[3,43; 10,55]',
     'Hit@5 (%)': 10.50, 'MRR': 0.0805, 'Reachable (%)': 9.32, 'Out-of-reach (%)': 0.00},
    {'Hệ thống': 'Phi-3.5-mini closed',    'Hit@1 (%)': 2.21, 'KTC 95%': '[0,86; 5,54]',
     'Hit@5 (%)': 6.08,  'MRR': 0.0351, 'Reachable (%)': 3.39, 'Out-of-reach (%)': 0.00},
    {'Hệ thống': 'BM25-only (đóng)',       'Hit@1 (%)': 1.66, 'KTC 95%': '[0,57; 4,76]',
     'Hit@5 (%)': 9.39,  'MRR': 0.0533, 'Reachable (%)': 2.54, 'Out-of-reach (%)': 0.00},
    {'Hệ thống': 'KG-only',                'Hit@1 (%)': 1.66, 'KTC 95%': '[0,57; 4,76]',
     'Hit@5 (%)': 6.08,  'MRR': 0.0335, 'Reachable (%)': 2.54, 'Out-of-reach (%)': 0.00},
    {'Hệ thống': 'BM25 tự do (13.081 mã)', 'Hit@1 (%)': 8.29, 'KTC 95%': '[5,09; 13,22]',
     'Hit@5 (%)': 14.92, 'MRR': 0.1128, 'Reachable (%)': 9.32, 'Out-of-reach (%)': 6.35},
    {'Hệ thống': 'LLM tự do (Meta-Llama-3-8B-Instruct)', 'Hit@1 (%)': 12.71,
     'KTC 95%': '[8,62; 18,35]', 'Hit@5 (%)': 14.36, 'MRR': 0.1358,
     'Reachable (%)': 17.80, 'Out-of-reach (%)': 3.17},
    {'Hệ thống': 'LLM tự do (Qwen2.5-7B-Instruct)', 'Hit@1 (%)': 7.73,
     'KTC 95%': '[4,66; 12,56]', 'Hit@5 (%)': 15.47, 'MRR': 0.1076,
     'Reachable (%)': 10.17, 'Out-of-reach (%)': 3.17},
    {'Hệ thống': 'LLM tự do (Phi-3.5-mini-instruct)', 'Hit@1 (%)': 7.73,
     'KTC 95%': '[4,66; 12,56]', 'Hit@5 (%)': 11.60, 'MRR': 0.0930,
     'Reachable (%)': 7.63, 'Out-of-reach (%)': 7.94},
    {'Hệ thống': f'LLM tự do ({MODEL_ID.split("/")[-1]})',
     'Hit@1 (%)': summary['hit1_pct'], 'KTC 95%': summary['hit1_ci'],
     'Hit@5 (%)': summary['hit5_pct'], 'MRR': summary['mrr'],
     'Reachable (%)': summary['hit1_reachable_pct'],
     'Out-of-reach (%)': summary['hit1_outreach_pct']},
])
bang.to_csv(f'{OUT}/tn4_bang12_tong_hop.csv', index=False, encoding='utf-8-sig')
print(bang.to_string(index=False))

## 14. Ghi chú diễn giải — sinh sẵn `.docx`

Mục 9.3 phiếu yêu cầu một tệp `.docx` 1–2 trang trả lời 5 câu, kèm 3 ví dụ ca.
Cell dưới điền sẵn cả 5 câu bằng số vừa đo và chọn đúng 3 ca ví dụ.
**Đọc lại và viết thêm nhận định trước khi gửi** — phần diễn giải là việc của người,
cell này chỉ lo phần số cho khỏi chép tay sai.

In [ ]:
from docx import Document
from docx.shared import Pt

# --- 3 vi du ca: b (chi PhoGPT dung), c (chi MedKG-HRR dung), ca hai dung ---
medkg_hit = (med['rank3'] == 1).astype(int)
u = u_hit
ca_b    = [i for i in med.index if u[i] == 1 and medkg_hit[i] == 0]
ca_c    = [i for i in med.index if u[i] == 0 and medkg_hit[i] == 1]
ca_hai  = [i for i in med.index if u[i] == 1 and medkg_hit[i] == 1]

def mo_ta_ca(cid):
    r = mine.loc[cid]
    return (f"{cid} | gold3={r['gold3']} ({med.loc[cid,'gold']}) | "
            f"out-of-reach={not r['voi_toi_duoc']}\n"
            f"    Hỏi     : {str(med.loc[cid,'query_text'])[:200]}\n"
            f"    PhoGPT  : {str(r['top_10_3char'])[:120]}\n"
            f"    MedKG   : top1={med.loc[cid,'top1_ma']} "
            f"({med.loc[cid,'top1_ten_benh']}), rank3={med.loc[cid,'rank3']}")

mc_181 = df_mc.iloc[0]; mc_118 = df_mc.iloc[1]

doc = Document()
doc.add_heading('TN4 — Ghi chú diễn giải: PhoGPT-4B-Chat unconstrained', 0)
doc.add_paragraph(f"Model: {MODEL_ID} | {summary['so_luong_bit']} | "
                  f"n = 181 ca độc lập | catalogue 13.081 mã ICD-10")
doc.add_paragraph(f"Thời gian chạy {summary['phut_chay']} phút, VRAM đỉnh "
                  f"{summary['vram_dinh_GB']} GB. SHA-256 prompt: {PROMPT_SHA[:16]}…")

doc.add_heading('1. Hit@1 tổng và kịch bản', level=1)
doc.add_paragraph(
    f"Hit@1 = {summary['hit1_pct']}% (KTC 95% {summary['hit1_ci']}), "
    f"Hit@5 = {summary['hit5_pct']}%, MRR = {summary['mrr']}. "
    f"Rơi vào KỊCH BẢN {KICH_BAN} theo mục 1 của phiếu.")

doc.add_heading('2. Xếp hạng trong nhóm hệ tự do của Bảng 12', level=1)
xh = sorted([('PhoGPT-4B-Chat', summary['hit1_pct']),
             ('Llama-3-8B-Instruct', 12.71), ('Qwen2.5-7B-Instruct', 7.73),
             ('Phi-3.5-mini-instruct', 7.73), ('BM25 tự do', 8.29)],
            key=lambda x: -x[1])
for i, (ten, v) in enumerate(xh, 1):
    doc.add_paragraph(f"{i}. {ten}: {v}%", style='List Number' if i == 1 else None)
doc.add_paragraph("PhoGPT xếp thứ "
                  f"{[t for t,_ in xh].index('PhoGPT-4B-Chat')+1}/5 trong nhóm không ràng buộc danh mục.")

doc.add_heading('3. Nhóm 63 ca out-of-reach', level=1)
doc.add_paragraph(
    f"PhoGPT bắt đúng {int(outr['hit1'].sum())}/63 ca = "
    f"{summary['hit1_outreach_pct']}% (KTC 95% {summary['hit1_outreach_ci']}). "
    f"Mốc so sánh: Phi-3.5-mini 5/63 = 7,94% (cao nhất hiện có), "
    f"Llama-3 2/63 = 3,17%, BM25 tự do 4/63 = 6,35%, "
    f"năm hệ đóng danh mục 0,00% theo thiết kế.")

doc.add_heading('4. Tỷ lệ mã hợp lệ trong catalogue 13.081 mã', level=1)
doc.add_paragraph(
    f"Trung bình {summary['avg_n_valid_in_top10']}/10 mã mỗi ca nằm trong catalogue; "
    f"tính trên tổng số mã parse được là {summary['pct_ma_hop_le']}%. "
    f"Số ca parse ra 0 mã: {(df_out['n_ma_parse_duoc'] == 0).sum()}. "
    f"Mốc Llama-3: 9,02/10 mã, 90,22% hợp lệ.")

doc.add_heading('5. McNemar so với MedKG-HRR', level=1)
doc.add_paragraph(
    f"n=181: b={mc_181['b_chi_phogpt_dung']}, c={mc_181['c_chi_he_kia_dung']}, "
    f"p = {mc_181['mcnemar_p']}.")
doc.add_paragraph(
    f"n=118 (chỉ reachable): b={mc_118['b_chi_phogpt_dung']}, "
    f"c={mc_118['c_chi_he_kia_dung']}, p = {mc_118['mcnemar_p']}. "
    f"Mốc Llama-3 trên cùng tập con: b=20, c=5, p=0,0041.")

doc.add_heading('6. Ba ví dụ định tính', level=1)
for nhan, ds in [('(b) Chỉ PhoGPT đúng', ca_b),
                 ('(c) Chỉ MedKG-HRR đúng', ca_c),
                 ('Cả hai cùng đúng', ca_hai)]:
    doc.add_heading(nhan, level=2)
    doc.add_paragraph(mo_ta_ca(ds[0]) if ds else 'Không có ca nào thuộc nhóm này.')

doc.add_heading('7. Nhận định (người viết bổ sung)', level=1)
doc.add_paragraph('[…]')

F_DOCX = f'{OUT}/tn4_ghichu_diengiai.docx'
doc.save(F_DOCX)
print('Da sinh:', F_DOCX)
print(f"  ca b (chi PhoGPT dung)  : {len(ca_b)}")
print(f"  ca c (chi MedKG-HRR dung): {len(ca_c)}")
print(f"  ca ca hai dung           : {len(ca_hai)}")

## 15. Băm SHA-256 và đóng gói

Phiếu mục 9.1 yêu cầu 5 tệp. Gói dưới có đủ 5 tệp đó cộng hai tệp bổ sung
(`tn4_theo_chuong.csv`, `tn4_bang12_tong_hop.csv`) để xếp cạnh TN3 được.

In [ ]:
import hashlib, shutil

BAT_BUOC = ['tn4_perquery_phogpt_unconstrained.csv', 'tn4_summary.csv',
            'tn4_mcnemar.csv', 'tn4_ghichu_diengiai.docx']

lines = []
for f in sorted(os.listdir(OUT)):
    if f.endswith('SHA256.txt'):
        continue
    with open(f'{OUT}/{f}', 'rb') as fh:
        lines.append(f'{hashlib.sha256(fh.read()).hexdigest()}  {f}')
with open(f'{OUT}/tn4_SHA256.txt', 'w', encoding='utf-8', newline='\n') as fh:
    fh.write('\n'.join(lines) + '\n')
print('\n'.join(lines))

thieu = [f for f in BAT_BUOC if not os.path.exists(f'{OUT}/{f}')]
print('\nDU 5 TEP BAT BUOC (tn4_SHA256.txt la tep thu 5).' if not thieu
      else f'\nTHIEU: {thieu}')

shutil.make_archive('/content/TN4_KetQua', 'zip', OUT)
from google.colab import files
files.download('/content/TN4_KetQua.zip')

## 16. Checklist trước khi gửi

Gửi `TN4_KetQua_[ngày].zip` tới **tiennt@ut.edu.vn**, subject
`[TN4 MedicalGraph] Kết quả PhoGPT unconstrained`.

**Báo ngay thầy, không đợi chạy xong** (mục 10.1 phiếu), nếu:

| Điều kiện | Lý do |
|---|---|
| Dry run Hit@1 < 1/10 sau 3 lần thử prompt | Model không hiểu task |
| Trung bình mã hợp lệ < 3/10 | Chế mã giả liên tục |
| Hit@1 full ≥ 15% | Kịch bản C — phải reframe Finding C |
| Hit@1 full = 0% | Nghi bug |
| Chạy > 4 giờ cho 181 ca | Quota T4 sắp hết |
| CUDA OOM kể cả 4-bit | Cần Colab Pro hoặc Kaggle |

**Ghi lại khi báo:** `MODEL_ID`, fp16 hay 4-bit, `max_new_tokens`, VRAM đỉnh,
số ca parse được 0 mã, và bốn con số: Hit@1 tổng, Hit@1 out-of-reach, % mã hợp lệ,
McNemar p (n=118) so với MedKG-HRR.